#Aggregation by Rishab Kumar
###Here the data is prepared for the analysis

In [0]:
%sql
USE CATALOG dbacademy;
USE SCHEMA dbuser_100426;

In [0]:
menu_df = spark.table('silver_menu_items_table')
restaurant_df = spark.table('silver_restaurants_table')
# display(menu_df.limit(10))
# display(restaurant_df.limit(10))

In [0]:
"""
Potential Use Cases
* Price analysis: Compare food prices across Indian cities, cuisines, and restaurant tiers
* Cuisine mapping: Analyze cuisine diversity and regional food preferences
* Rating analysis: Study rating distributions, correlations with price, delivery time, and cuisine type
* NLP on menus: Train models on 12.8M+ menu item descriptions
* Geospatial analysis: Map restaurant density and food delivery landscape across India
* Market research: Understand Swiggy penetration across tier-1, tier-2, and tier-3 cities
* Veg vs Non-Veg analysis: Study vegetarian food availability patterns across regions
"""

'\nPotential Use Cases\n* Price analysis: Compare food prices across Indian cities, cuisines, and restaurant tiers\n* Cuisine mapping: Analyze cuisine diversity and regional food preferences\n* Rating analysis: Study rating distributions, correlations with price, delivery time, and cuisine type\n* NLP on menus: Train models on 12.8M+ menu item descriptions\n* Geospatial analysis: Map restaurant density and food delivery landscape across India\n* Market research: Understand Swiggy penetration across tier-1, tier-2, and tier-3 cities\n* Veg vs Non-Veg analysis: Study vegetarian food availability patterns across regions\n'

In [0]:
# menu_df.columns
"""
['restaurant_id',
 'restaurant_name',
 'city_name',
 'city_slug',
 'category',
 'item_id',
 'item_name',
 'price',
 'description',
 'is_veg',
 'rating',
 'rating_count',
 'in_stock',
 'has_addons',
 'has_variants',
 'source']
"""
# restaurant_df.columns
"""
['restaurant_id',
 'name',
 'area',
 'locality',
 'city_name',
 'city_slug',
 'latitude',
 'longitude',
 'cuisines',
 'avg_rating',
 'total_ratings',
 'cost_for_two',
 'delivery_time_mins',
 'is_veg',
 'is_open',
 'menu_category_count',
 'menu_item_count',
 'source']
"""

"\n['restaurant_id',\n 'name',\n 'area',\n 'locality',\n 'city_name',\n 'city_slug',\n 'latitude',\n 'longitude',\n 'cuisines',\n 'avg_rating',\n 'total_ratings',\n 'cost_for_two',\n 'delivery_time_mins',\n 'is_veg',\n 'is_open',\n 'menu_category_count',\n 'menu_item_count',\n 'source']\n"

In [0]:
# menu_df.select("city_name").count() - menu_df.select("city_slug").count()
# menu_df.select("city_name").distinct().count()

# restaurant_df.select("city_name").count() - restaurant_df.select("city_slug").count()
restaurant_df.select("city_name").distinct().count()
# restaurant_df.select("area").distinct().count()
# restaurant_df.select("locality").distinct().count()


565

In [0]:


#price analysis by cities from menu table
menu_df.select(\
    ["restaurant_id","price", "city_name"])\
        .createOrReplaceTempView("select_menu_attributes")

query_menu_price_by_city = """
SELECT city_name, 
round(avg(price), 2) as avg_price, 
min(price) as min_price, 
max(price) as max_price, 
count(*) as total_items 
FROM select_menu_attributes 
GROUP BY city_name 
ORDER BY total_items DESC
"""
df_menu_price_by_city = spark.sql(query_menu_price_by_city)
# display(df_menu_price_by_city.count())
display(df_menu_price_by_city.limit(10))



# price analysis by cities from restaurant table
restaurant_df.select(\
    ["restaurant_id", "area", "locality", "city_name", "cuisines", "cost_for_two"])\
        .createOrReplaceTempView("select_restaurant_attributes")

query_restaurant_price_by_city = """
SELECT city_name, 
round(avg(cost_for_two), 2) as avg_cost_for_two, 
min(cost_for_two) as min_cost_for_two, 
max(cost_for_two) as max_cost_for_two,
count(*) as total_restaurants 
FROM select_restaurant_attributes 
GROUP BY city_name 
ORDER BY total_restaurants DESC
"""
df_restaurant_price_by_city = spark.sql(query_restaurant_price_by_city)
# display(df_restaurant_price_by_city.count())
display(df_restaurant_price_by_city.limit(10))


query_price_analysis_by_cuisine = """
SELECT cuisines, 
round(avg(cost_for_two), 2) as avg_cost_for_two, 
count(*) as total_restaurants 
FROM select_restaurant_attributes 
GROUP BY cuisines 
ORDER BY total_restaurants DESC
"""
df_price_analysis_by_cuisine = spark.sql(query_price_analysis_by_cuisine)
# display(df_price_analysis_by_cuisine.count())
display(df_price_analysis_by_cuisine.limit(10))


query_cuisine_by_city = """
WITH ranked_records AS (
    SELECT city_name, 
    cuisines, 
    count(*) as total_restaurant,
    ROW_NUMBER() over (PARTITION BY city_name ORDER BY count(*) DESC) as cuisine_row_num
    FROM select_restaurant_attributes 
    GROUP BY city_name, cuisines )
SELECT city_name, 
cuisines as top_cuisines,
total_restaurant
FROM ranked_records
WHERE cuisine_row_num =1
ORDER BY total_restaurant DESC
"""
df_cuisine_by_city = spark.sql(query_cuisine_by_city)
# display(df_cuisine_by_city.count())
display(df_cuisine_by_city.limit(10))


query_city_by_cuisine = """
WITH ranked_records AS (
    SELECT cuisines, 
    city_name, 
    count(*) as total_restaurant,
    ROW_NUMBER() over (PARTITION BY cuisines ORDER BY count(*) DESC) as city_row_num
    FROM select_restaurant_attributes 
    GROUP BY cuisines, city_name )
SELECT cuisines, 
city_name as top_cities,
total_restaurant
FROM ranked_records
WHERE city_row_num = 1
ORDER BY total_restaurant DESC
"""
df_city_by_cuisine = spark.sql(query_city_by_cuisine)
# display(df_city_by_cuisine.count())
display(df_city_by_cuisine.limit(10))




city_name,avg_price,min_price,max_price,total_items
central goa,294.20,1.90,5999.00,171411
hajipur,222.04,1.79,5999.00,165127
mumbai,279.63,1.15,11499.00,161478
lakshadweep,239.52,1.50,5200.00,159837
amritsar,288.43,1.30,7000.00,148154
jaipur,226.37,1.79,4900.00,147688
patna,258.14,1.79,4999.00,146668
delhi,283.07,1.79,8500.00,142315
dehradun,239.48,1.79,14500.00,138494
mughalsarai,235.97,1.25,5075.00,132326


city_name,avg_cost_for_two,min_cost_for_two,max_cost_for_two,total_restaurants
indore,264.62,50,1600,2000
hyderabad,327.06,100,2000,2000
dehradun,318.97,30,10000,2000
kanpur,261.50,80,2200,2000
allahabad,257.63,99,1500,2000
bharabanki,261.34,50,2800,2000
central goa,484.01,100,3000,2000
ahmedabad,324.75,50,2000,2000
hajipur,282.51,99,2000,2000
kochi,290.03,59,10000,2000


cuisines,avg_cost_for_two,total_restaurants
chinese,278.85,29626
north indian,284.42,14349
biryani,298.16,13186
indian,275.72,11386
south indian,241.82,9149
pizzas,272.71,8888
bakery,301.74,7710
fast food,360.15,7294
beverages,252.37,6908
snacks,236.65,6152


city_name,top_cuisines,total_restaurant
mughalsarai,chinese,680
kanchrapara,chinese,539
hajipur,chinese,526
dehradun,chinese,500
allahabad,chinese,408
kanpur,chinese,397
bharabanki,chinese,395
central goa,chinese,376
bagdogra,chinese,375
delhi,chinese,373


cuisines,top_cities,total_restaurant
chinese,mughalsarai,680
south indian,coimbatore,370
north indian,jaipur,339
biryani,coimbatore,335
indian,bharabanki,270
bakery,kota,229
kerala,kochi,221
pizzas,kanpur,216
goan,central goa,203
maharashtrian,nashik,199


In [0]:
# saving the dataframes as gold tables
df_menu_price_by_city.write.mode("overwrite").saveAsTable("gold_menu_price_by_city")
df_restaurant_price_by_city.write.mode("overwrite").saveAsTable("gold_restaurant_price_by_city")
df_price_analysis_by_cuisine.write.mode("overwrite").saveAsTable("gold_price_analysis_by_cuisine")
df_cuisine_by_city.write.mode("overwrite").saveAsTable("gold_cuisine_by_city")
df_city_by_cuisine.write.mode("overwrite").saveAsTable("gold_city_by_cuisine")